# EWS Anomaly Detection — Full Pipeline Simulation

1. Oracle'a veri yukle
2. Oracle'dan oku, model egit, stabilite olc
3. Skorla, Oracle'a yaz
4. Validation + alert raporu

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

from engine.config_loader import load_config, load_secrets, get_feature_list, get_label, get_alert_bands
from engine.oracle_io import OracleConnector
from engine.models import AnomalyModels
from engine.scorer import AnomalyScorer
from generate_data import generate_training_data, generate_scoring_data

config = load_config()
features = get_feature_list(config)
print(f'Feature sayisi: {len(features)}')
print(f'Katmanlar: instant({len(config["features"]["instant"])}), rolling_4w({len(config["features"]["rolling_4w"])}), trend({len(config["features"]["trend"])})')

---
## 1. Veri Uret ve Oracle'a Yukle

In [ ]:
from engine.pipeline import EWSPipeline

pipe = EWSPipeline()

train_df = generate_training_data()
scoring_df, labels = generate_scoring_data()

print(f'Training: {train_df.shape}, split: {train_df["split_flag"].value_counts().to_dict()}')
print(f'Scoring: {scoring_df.shape}, anomalies: {labels["is_anomaly"].sum()}')

pipe.load_data(train_df, scoring_df)

---
## 2. Oracle'dan Oku + EDA

In [ ]:
secrets = load_secrets()
ora = OracleConnector(config, secrets)
ora.connect()

train_ora = ora.read_training_data(split='TRAIN')
test_ora = ora.read_training_data(split='TEST')
print(f'Oracle Train: {train_ora.shape}, Test: {test_ora.shape}')

In [ ]:
# Temel feature dagilimlar
key_feats = ['utilization_ratio', 'dpd_current', 'min_payment_only_count_4w',
             'payment_to_min_ratio_4w', 'outstanding_balance', 'checking_balance']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, feat in zip(axes.flat, key_feats):
    ax.hist(train_ora[feat].dropna(), bins=50, alpha=0.7, color='#3498db', edgecolor='white')
    ax.set_title(get_label(config, feat), fontsize=10)
    ax.axvline(train_ora[feat].mean(), color='red', linestyle='--', alpha=0.7)
fig.suptitle('Training Data — Temel Feature Dagilimlari', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Korelasyon matrisi
corr = train_ora[features].corr()
fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.3, ax=ax,
            xticklabels=[f[:14] for f in features],
            yticklabels=[f[:14] for f in features])
ax.set_title('Feature Korelasyon Matrisi', fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. Model Egitimi + Stabilite Testi

In [ ]:
models = AnomalyModels(config)
models.fit(train_ora[features].fillna(0).values)

In [ ]:
# Reconstruction Stability: Train vs Test
from scipy.stats import ks_2samp

X_tr = models.transform(train_ora[features].fillna(0).values)
X_te = models.transform(test_ora[features].fillna(0).values)

tr_err = models._ae_total_error(X_tr)
te_err = models._ae_total_error(X_te)
ks_stat, ks_pval = ks_2samp(tr_err, te_err)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(tr_err, bins=60, alpha=0.6, color='#3498db', label=f'Train (n={len(tr_err)})', density=True, edgecolor='white')
axes[0].hist(te_err, bins=60, alpha=0.6, color='#e74c3c', label=f'Test (n={len(te_err)})', density=True, edgecolor='white')
axes[0].set_title(f'AE Reconstruction Error\nKS p={ks_pval:.4f}', fontsize=12)
axes[0].legend()

tr_md = models._mahal_distances(X_tr)
te_md = models._mahal_distances(X_te)
ks2, ks2p = ks_2samp(tr_md, te_md)
axes[1].hist(tr_md, bins=60, alpha=0.6, color='#3498db', label='Train', density=True, edgecolor='white')
axes[1].hist(te_md, bins=60, alpha=0.6, color='#e74c3c', label='Test', density=True, edgecolor='white')
axes[1].set_title(f'Mahalanobis Distance\nKS p={ks2p:.4f}', fontsize=12)
axes[1].legend()
plt.tight_layout()
plt.show()

ratio = te_err.mean() / tr_err.mean()
print(f'AE loss ratio (test/train): {ratio:.3f}x')
print(f'Reconstruction KS: p={ks_pval:.4f} {"OK" if ks_pval > 0.05 else "UYARI"}')

In [ ]:
# Score Distribution: Train vs Test
scorer = AnomalyScorer(config, models)
train_results = scorer.score(train_ora)
test_results = scorer.score(test_ora)

ks3, ks3p = ks_2samp(train_results['anomaly_score'], test_results['anomaly_score'])

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(train_results['anomaly_score'], bins=50, alpha=0.5, color='#3498db', label='Train', density=True, edgecolor='white')
ax.hist(test_results['anomaly_score'], bins=50, alpha=0.5, color='#e74c3c', label='Test', density=True, edgecolor='white')

bands = get_alert_bands(config)
band_colors = {'SARI': '#f1c40f', 'TURUNCU': '#e67e22', 'KIRMIZI': '#e74c3c'}
for bname, (lo, hi) in bands.items():
    if bname in band_colors:
        ax.axvline(lo, color=band_colors[bname], linestyle='--', alpha=0.6)

ax.set_title(f'Ensemble Score: Train vs Test (KS p={ks3p:.4f})', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

print('Bant karsilastirmasi:')
for band in ['KIRMIZI','TURUNCU','SARI','NORMAL']:
    tr_pct = (train_results['alert_band']==band).mean()*100
    te_pct = (test_results['alert_band']==band).mean()*100
    print(f'  {band:10s}  Train: %{tr_pct:.1f}  Test: %{te_pct:.1f}')

---
## 4. Scoring — Bugunun Verisi

In [ ]:
scoring_ora = ora.read_scoring_data()
print(f'Scoring verisi: {scoring_ora.shape}')

results = scorer.score(scoring_ora)
print(f'\nBant dagilimi:')
print(results['alert_band'].value_counts().sort_index())

In [ ]:
# Skor dagilimi
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'NORMAL':'#2ecc71', 'SARI':'#f1c40f', 'TURUNCU':'#e67e22', 'KIRMIZI':'#e74c3c'}
for band in ['NORMAL','SARI','TURUNCU','KIRMIZI']:
    s = results[results['alert_band']==band]['anomaly_score']
    axes[0].hist(s, bins=40, alpha=0.6, color=colors[band], label=f'{band} ({len(s)})', edgecolor='white')
axes[0].set_title('Anomaly Score Dagilimi', fontsize=12)
axes[0].legend()

axes[1].scatter(results['ae_score'], results['if_score'], c=results['anomaly_score'],
                cmap='RdYlGn_r', alpha=0.4, s=15)
axes[1].set_xlabel('AE Score')
axes[1].set_ylabel('IF Score')
axes[1].set_title('AE vs IF', fontsize=12)
plt.colorbar(axes[1].collections[0], ax=axes[1], label='Ensemble')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 alert
print('='*80)
print('TOP 10 ALERT')
print('='*80)

for _, row in results.head(10).iterrows():
    print(f"\n  {row['customer_id']}  |  Skor: {row['anomaly_score']}  |  {row['alert_band']}  |  flags: {row['uni_flag_count']}")
    print(f"  AE: {row['ae_score']}  IF: {row['if_score']}  MD: {row['md_score']}")
    for feat, d in row['detay'].items():
        ico = 'UP' if d['degisim_pct'] > 0 else 'DN'
        print(f"    {d['label']}: {d['beklenen']} -> {d['gerceklesen']} ({ico}%{abs(d['degisim_pct']):.0f}, katki%{d['katki_pct']:.0f})")

---
## 5. Sonuclari Oracle'a Yaz

In [ ]:
from datetime import date

scoring_date = date.today()
pipe._load_model = lambda: None  # modeli zaten yukledik
pipe.models = models
pipe.scorer = scorer
pipe._write_to_oracle(ora, results, scoring_date)

print('Oracle yazildi')

In [ ]:
# Dogrulama
verify = ora._read_query(f"""
    SELECT CUSTOMER_ID, ANOMALY_SCORE, ALERT_BAND, REASON_1
    FROM {ora._qualified_table_name('results')}
    WHERE ALERT_BAND = 'KIRMIZI'
    ORDER BY ANOMALY_SCORE DESC
    FETCH FIRST 5 ROWS ONLY
""")
print(f'Oracle KIRMIZI alertler ({len(verify)}):')
verify

---
## 6. Validation — Enjekte Edilen Anomalileri Yakaliyor muyuz?

In [ ]:
eval_df = results.merge(labels, on='customer_id')
alerted = eval_df[eval_df['alert_band'].isin(['KIRMIZI','TURUNCU','SARI'])]

print('='*65)
print('VALIDATION')
print('='*65)

for band in ['KIRMIZI','TURUNCU','SARI']:
    ib = eval_df[eval_df['alert_band']==band]
    n = len(ib)
    tp = ib['is_anomaly'].sum()
    prec = tp/n*100 if n > 0 else 0
    print(f'\n  {band}: {n} alert, {tp} gercek (precision: %{prec:.1f})')

print(f'\n  RECALL:')
for t in ['A_UNIVARIATE','B_MULTIVARIATE','C_SUBTLE_DRIFT']:
    tot = (eval_df['anomaly_type']==t).sum()
    c = (alerted['anomaly_type']==t).sum()
    print(f'    {t}: {c}/{tot} (%{c/tot*100:.0f})')
ta = eval_df['is_anomaly'].sum()
tc = alerted['is_anomaly'].sum()
print(f'    TOPLAM: {tc}/{ta} (%{tc/ta*100:.0f})')

In [ ]:
# Anomali tipine gore skor dagilimi
fig, ax = plt.subplots(figsize=(12, 5))
tc = {'NORMAL':'#2ecc71','A_UNIVARIATE':'#e74c3c','B_MULTIVARIATE':'#e67e22','C_SUBTLE_DRIFT':'#9b59b6'}
for atype, color in tc.items():
    s = eval_df[eval_df['anomaly_type']==atype]['anomaly_score']
    ax.hist(s, bins=40, alpha=0.5, color=color, label=f'{atype} (n={len(s)})', density=True)

for bname, (lo, hi) in get_alert_bands(config).items():
    if bname in band_colors:
        ax.axvline(lo, color=band_colors[bname], linestyle='--', alpha=0.6)

ax.set_title('Anomali Tipi vs Skor Dagilimi', fontsize=13)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Precision vs Recall esik analizi
thresholds = list(range(40, 100, 5))
total_anomalies = eval_df['is_anomaly'].sum()
pr_data = []
for t in thresholds:
    flagged = eval_df[eval_df['anomaly_score'] >= t]
    n_f = len(flagged)
    n_t = flagged['is_anomaly'].sum()
    pr_data.append({
        'Esik': t,
        'Alert': n_f,
        'Dogru': n_t,
        'Precision': round(n_t/n_f*100, 1) if n_f > 0 else 0,
        'Recall': round(n_t/total_anomalies*100, 1),
    })

pr_df = pd.DataFrame(pr_data)
print(pr_df.to_string(index=False))

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(pr_df['Esik'], pr_df['Precision'], 'o-', color='#e74c3c', linewidth=2, label='Precision')
ax1.plot(pr_df['Esik'], pr_df['Recall'], 's-', color='#3498db', linewidth=2, label='Recall')
ax1.set_xlabel('Esik')
ax1.set_ylabel('%')
ax1.set_title('Precision vs Recall', fontsize=13)
ax1.legend()
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. Deep Dive — Tek Musteri

In [ ]:
# KIRMIZI + gercek anomali olan bir musteri sec
red = eval_df[(eval_df['alert_band']=='KIRMIZI') & (eval_df['is_anomaly'])]
sample = red.iloc[0]
cust_id = sample['customer_id']

print(f'Musteri: {cust_id}')
print(f'Tip: {sample["anomaly_type"]}')
print(f'Skor: {sample["anomaly_score"]} | Bant: {sample["alert_band"]} | Flags: {sample["uni_flag_count"]}')
print(f'AE: {sample["ae_score"]} | IF: {sample["if_score"]} | MD: {sample["md_score"]}')
print(f'\nNedenler:')
for feat, d in sample['detay'].items():
    ico = 'UP' if d['degisim_pct'] > 0 else 'DN'
    print(f'  {d["label"]}: {d["beklenen"]} -> {d["gerceklesen"]} ({ico}%{abs(d["degisim_pct"]):.0f}, katki%{d["katki_pct"]:.0f})')

In [ ]:
# 3 model katki paylari
cust_row = scoring_ora[scoring_ora['customer_id']==cust_id]
X_c = models.transform(cust_row[features].fillna(0).values)

ae_c = models.ae_contribution(X_c)[0]
if_c = models.if_contribution(X_c)[0]
md_c = models.md_contribution(X_c)[0]
unified = 0.5*ae_c + 0.3*if_c + 0.2*md_c

contrib = pd.DataFrame({
    'Feature': features,
    'Label': [get_label(config, f) for f in features],
    'AE%': (ae_c*100).round(1),
    'IF%': (if_c*100).round(1),
    'MD%': (md_c*100).round(1),
    'Birlesik%': (unified*100).round(1),
}).sort_values('Birlesik%', ascending=False)

print(f'\nTop 8 Feature Katki:')
print(contrib.head(8).to_string(index=False))

# Bar chart
top8 = contrib.head(8)
fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(top8))
w = 0.22
ax.bar(x-w, top8['AE%'], w, label='AE', color='#3498db', alpha=0.8)
ax.bar(x, top8['IF%'], w, label='IF', color='#e67e22', alpha=0.8)
ax.bar(x+w, top8['MD%'], w, label='MD', color='#9b59b6', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([l[:18] for l in top8['Label']], rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Katki %')
ax.set_title(f'{cust_id} - 3 Model Feature Katkilari', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Beklenen vs Gerceklesen
expected_X = models.ae_reconstruct(X_c)
top5 = contrib.head(5)['Feature'].values

act_vals, exp_vals, feat_labels = [], [], []
for feat in top5:
    j = features.index(feat)
    act = X_c[0, j] * models.scaler.scale_[j] + models.scaler.mean_[j]
    exp = expected_X[0, j] * models.scaler.scale_[j] + models.scaler.mean_[j]
    act_vals.append(act)
    exp_vals.append(exp)
    feat_labels.append(get_label(config, feat)[:18])

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(feat_labels))
ax.bar(x-0.2, exp_vals, 0.35, label='Beklenen (AE)', color='#3498db', alpha=0.7)
ax.bar(x+0.2, act_vals, 0.35, label='Gerceklesen', color='#e74c3c', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(feat_labels, rotation=25, ha='right', fontsize=9)
ax.set_title(f'{cust_id} - Beklenen vs Gerceklesen', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

---
## 8. Excel Raporu

In [ ]:
import os
os.makedirs('output', exist_ok=True)

alert_df = results[results['alert_band'].isin(['KIRMIZI','TURUNCU','SARI'])].copy()
report_rows = []
for _, row in alert_df.iterrows():
    r = {'Musteri': row['customer_id'], 'Skor': row['anomaly_score'], 'Bant': row['alert_band'],
         'AE': row['ae_score'], 'IF': row['if_score'], 'MD': row['md_score'], 'Flags': row['uni_flag_count']}
    for i, (feat, d) in enumerate(row['detay'].items(), 1):
        ico = 'UP' if d['degisim_pct'] > 0 else 'DN'
        r[f'Neden_{i}'] = f"{d['label']}: {d['beklenen']}->{d['gerceklesen']} ({ico}%{abs(d['degisim_pct']):.0f})"
    report_rows.append(r)

report = pd.DataFrame(report_rows)
report.to_excel('output/haftalik_alert_raporu.xlsx', index=False, sheet_name='EWS Alerts')
print(f'Kaydedildi: output/haftalik_alert_raporu.xlsx ({len(report)} alert)')
report.head(10)

In [ ]:
# Temizlik
ora.close()
print('Oracle baglantisi kapatildi')